In [0]:
dbutils.widgets.text("table_row","""{"table_name":"credit_bureau_reports","source_system":"blob","source_schema":"dummy","source_path":"/Volumes/banking/source/volume/credit_bureau_reports/"}""")
string_of_items_to_read=dbutils.widgets.get("table_row")
import json
tables_list=json.loads(string_of_items_to_read)
print(f"Table name involved in this op: {tables_list["table_name"]}")
print(f"Source system from where to read: {tables_list["source_system"]}")
print(f"Source path from where to read: {tables_list["source_path"]}")

table_name=tables_list["table_name"]
source_system=tables_list["source_system"]
source_path=tables_list["source_path"]


In [0]:
# Read the CSV as a streaming object

source_df=(
    spark.readStream
  .format("cloudFiles") 
  .option("cloudFiles.format", "csv") 
  .option("cloudFiles.schemaLocation", f"/Volumes/banking/source/volume/_schema/{table_name}")
  .option("header","true")
  .load(source_path)
)

In [0]:

spark.sql("USE CATALOG banking")
spark.sql("USE SCHEMA bronze")


In [0]:
credit_count = spark.sql("SELECT count(*) FROM banking.bronze.credit_bureau_reports").collect()[0][0]
payment_count = spark.sql("SELECT count(*) FROM banking.bronze.payment_gateway_logs").collect()[0][0]

# Print clean messages
print(f"📊 Credit Bureau Reports Count: {credit_count:,}")
print(f"💳 Payment Gateway Logs Count:  {payment_count:,}")

In [0]:


# Write the streaming dataframe into a checkpointed Delta Table
(
source_df.writeStream
.format("delta")
.option("checkpointLocation", f"/Volumes/banking/source/volume/_checkpoint/{table_name}")
.outputMode("append")
.option("schemaEvolutionMode","addNewColumns")
.trigger(availableNow=True) 
.toTable(f"bronze.{table_name}")    
)

In [0]:
credit_count = spark.sql("SELECT count(*) FROM banking.bronze.credit_bureau_reports").collect()[0][0]
payment_count = spark.sql("SELECT count(*) FROM banking.bronze.payment_gateway_logs").collect()[0][0]

# Print clean messages
print(f"After new load : Credit Bureau Reports Count: {credit_count:,}")
print(f"After new load :  Payment Gateway Logs Count:  {payment_count:,}")